# Practice Lab: Deepfakes, Watermarking & C2PA (Lesson 15.5)

Module 15 . 3 exercises . AI detection + watermark robustness + full pipeline


# Lesson 15.5: Deepfakes, Watermarking & C2PA

3 exercises: 1E + 1M + 1C

Module 15: Ethics, Safety, Governance & Explainability


In [ ]:
import hashlib, random, numpy as np
from datetime import datetime
from collections import Counter
random.seed(42)


---
## Exercise 1 (Easy): AI Text Detection


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np

class ATD:
    def burstiness(self,text):
        ss=[s.strip() for s in text.split(".") if len(s.strip())>5]
        if len(ss)<2: return 0.5
        ls=[len(s.split()) for s in ss]
        return round(np.std(ls)/max(np.mean(ls),1),3)
    def vocab(self,text):
        ws=[w.lower().strip(".,!?;:\'\"()") for w in text.split() if len(w)>1]
        if not ws: return 0.5
        return round(len(set(ws))/len(ws),3)
    def detect(self,text):
        b=self.burstiness(text); v=self.vocab(text); s=0
        if b<0.30: s+=0.40
        elif b<0.45: s+=0.15
        if v<0.50: s+=0.35
        elif v<0.60: s+=0.10
        return {"b":b,"v":v,"s":round(s,2),"pred":"AI" if s>=0.40 else "Human"}

d=ATD()
ai=["Machine learning is a branch of artificial intelligence. It enables systems to learn from data. Models improve over time. This approach has many applications.",
    "Natural language processing allows computers to understand text. It uses statistical methods and neural networks. Modern NLP systems achieve high accuracy. They can translate documents.",
    "Deep learning uses neural networks with multiple layers. Each layer extracts features. Training requires large datasets. The results have been impressive.",
    "Reinforcement learning trains agents through rewards. The agent explores environments. This approach succeeds in games. It also applies to robotics.",
    "Computer vision enables machines to interpret visuals. CNNs classify images. Object detection identifies items. These systems deploy in vehicles.",
    "Transfer learning allows models to leverage prior knowledge. Pre-trained models serve as starting points. Fine-tuning adapts to domains. This reduces data needs.",
    "Generative AI creates new content from training patterns. Large language models generate coherent text. Image models produce visuals. These tools transform workflows.",
    "Data preprocessing is essential in ML pipelines. It involves cleaning and transforming data. Feature engineering extracts information. Proper preprocessing impacts performance.",
    "Model evaluation uses metrics like accuracy and recall. Cross-validation estimates performance. The metric depends on the problem. Regular evaluation prevents overfitting.",
    "Cloud computing provides scalable AI infrastructure. GPU instances accelerate training. Serverless reduces overhead. Cost optimization matters for production."]
hu=["Ok so I spent 3 hours debugging this transformer and turns out? A TYPO. Three. Hours. Kill me lol",
    "The course was pretty good honestly. Some modules dragged but the projects were brilliant, especially RAG!",
    "Why does everyone say 'just use ChatGPT' like it solves everything? It hallucinates half the time!",
    "Started learning Python last month. Built a chatbot! It's terrible but hey, it works. Sort of.",
    "My prof said AI replaces all programmers. Meanwhile AI can't center a div. We're safe lmao",
    "Genuinely impressed by the attention explanation in lecture 7. Finally someone who doesn't assume a PhD!",
    "Hot take: most AI startups are just API wrappers with a nice UI. Change my mind. (Actually don't.)",
    "Tried to explain transformers to my mom. She thinks I work with robots that transform into cars.",
    "Been reading the original attention paper... the title aged like fine wine. They really called it.",
    "Ngl the Ollama setup was surprisingly easy? One command and boom, local LLM. The future is wild."]

all_t=[(t,"AI") for t in ai]+[(t,"Human") for t in hu]
tp=fp=tn=fn=0
print("AI Detection (20 samples):")
for i,(t,lb) in enumerate(all_t):
    r=d.detect(t); p=r["pred"]
    if p=="AI" and lb=="AI": tp+=1
    elif p=="AI" and lb=="Human": fp+=1
    elif p=="Human" and lb=="Human": tn+=1
    else: fn+=1
    if i<3 or i>=17 or p!=lb:
        print(f"  {i+1:>2} {lb:>5}/{p:<5} b={r['b']:.3f} v={r['v']:.3f} s={r['s']:.2f} {'Y' if p==lb else 'N'}")
acc=(tp+tn)/len(all_t); pr_=tp/max(tp+fp,1); rc=tp/max(tp+fn,1); f1=2*pr_*rc/max(pr_+rc,0.001)
print(f"\n  Acc={acc:.0%} Prec={pr_:.3f} Rec={rc:.3f} F1={f1:.3f} TP={tp} FP={fp} TN={tn} FN={fn}")


</details>


---
## Exercise 2 (Medium): Watermark Robustness


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib, random
random.seed(42)

class WM:
    def __init__(self,k="netsetos-2026"): self.k=k
    def _grn(self,p,c):
        return int(hashlib.sha256(f"{self.k}:{p}:{c}".encode()).hexdigest(),16)%2==1
    def detect(self,text):
        ws=text.lower().split()
        if len(ws)<5: return {"gr":0.5,"wm":False}
        g=sum(1 for i in range(1,len(ws)) if self._grn(ws[i-1],ws[i]))
        r=round(g/max(len(ws)-1,1),3)
        return {"gr":r,"wm":r>0.60}

def para(text,pct):
    ws=text.split(); reps={"is":"represents","the":"a","and":"plus","are":"remain",
        "can":"may","with":"using","for":"toward","has":"possesses","this":"that",
        "learning":"training","models":"systems","data":"information"}
    n=max(1,int(len(ws)*pct)); idx=random.sample(range(len(ws)),min(n,len(ws)))
    r=ws[:]
    for i in idx:
        c=r[i].lower().strip(".,!?;:")
        if c in reps: p=r[i][len(c):] if len(r[i])>len(c) else ""; r[i]=reps[c]+p
        else: r[i]=r[i][::-1]
    return " ".join(r)

wm=WM()
texts=["Machine learning is a powerful tool for data analysis and prediction",
    "The transformer architecture has revolutionized natural language processing",
    "Deep learning models can learn complex patterns from large datasets",
    "Reinforcement learning trains agents to make optimal decisions",
    "Computer vision enables machines to understand and interpret images",
    "Transfer learning allows knowledge to be shared across tasks",
    "Generative AI can create new content including text images and code",
    "Cloud computing provides scalable infrastructure for AI workloads",
    "Natural language processing helps computers understand human language",
    "Data preprocessing is essential for building effective ML pipelines"]

print("Watermark Robustness:")
# Original detection
od=sum(1 for t in texts if wm.detect(t)["wm"])
print(f"  Originals detected: {od}/{len(texts)}")

print(f"\n  {'Change%':>8} {'Survived':>10} {'Rate':>7}")
for pct in [0.10,0.30,0.50]:
    sv=0
    for t in texts:
        p=para(t,pct); r=wm.detect(p)
        if r["wm"]: sv+=1
    print(f"  {pct*100:>7.0f}% {sv:>8}/{len(texts)} {sv/len(texts)*100:>6.0f}%")

fp=sum(1 for t in texts if wm.detect(t)["wm"])
print(f"\n  Unwatermarked false positives: checked (same texts used)")


</details>


---
## Exercise 3 (Challenge): Full Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib, random, numpy as np
from datetime import datetime
random.seed(42)

class CAP:
    def __init__(self,k="netsetos-2026",cr="Netsetos"): self.k=k; self.cr=cr
    def _grn(self,p,c): return int(hashlib.sha256(f"{self.k}:{p}:{c}".encode()).hexdigest(),16)%2==1
    def wm_detect(self,text):
        ws=text.lower().split()
        if len(ws)<5: return {"gr":0.5,"wm":False}
        g=sum(1 for i in range(1,len(ws)) if self._grn(ws[i-1],ws[i]))
        r=round(g/max(len(ws)-1,1),3); return {"gr":r,"wm":r>0.60}
    def sign(self,content):
        ch=hashlib.sha256(content.encode()).hexdigest()
        sig=hashlib.sha256(f"sign:{ch}:{self.cr}:{self.k}".encode()).hexdigest()[:16]
        return {"claim":{"created":datetime.utcnow().isoformat()+"Z","creator":self.cr,
            "ai_generated":True},"content_hash":ch,"signature":sig}
    def verify_sig(self,content,manifest):
        return {"match":manifest["content_hash"]==hashlib.sha256(content.encode()).hexdigest()}
    def ai_detect(self,text):
        ss=[s.strip() for s in text.split(".") if len(s.strip())>5]
        b=round(np.std([len(s.split()) for s in ss])/max(np.mean([len(s.split()) for s in ss]),1),3) if len(ss)>=2 else 0.5
        ws=[w.lower().strip(".,!?") for w in text.split() if len(w)>1]
        v=round(len(set(ws))/max(len(ws),1),3); s=0
        if b<0.30: s+=0.40
        if v<0.50: s+=0.35
        return {"ai":s>=0.40}
    def full_verify(self,content,manifest):
        sg=self.verify_sig(content,manifest); wm=self.wm_detect(content); ai=self.ai_detect(content)
        if sg["match"] and wm["wm"]: return {"sig":True,"wm":True,"ai":ai["ai"],"conf":"HIGH","status":"AUTHENTIC"}
        elif sg["match"]: return {"sig":True,"wm":False,"ai":ai["ai"],"conf":"MEDIUM","status":"SIG VALID"}
        elif wm["wm"]: return {"sig":False,"wm":True,"ai":ai["ai"],"conf":"MEDIUM","status":"WM ONLY (modified)"}
        elif ai["ai"]: return {"sig":False,"wm":False,"ai":True,"conf":"LOW","status":"AI DETECTED (no provenance)"}
        return {"sig":False,"wm":False,"ai":False,"conf":"NONE","status":"NO AI SIGNALS"}

p=CAP()
contents=["Machine learning is a powerful approach. It enables pattern learning. Modern models achieve accuracy.",
    "NLP allows computers to understand text. Transformers dominate. They use attention mechanisms.",
    "Deep learning uses many layers. Each extracts features. Training needs compute.",
    "RL trains agents through trial. The agent receives rewards. It learns optimal strategies.",
    "Computer vision interprets visual data. CNNs excel at classification. Detection identifies items."]

signed=[{"c":c,"m":p.sign(c)} for c in contents]

print("Content Authenticity Pipeline:")

print("\n  Untampered:")
for i in range(3):
    r=p.full_verify(signed[i]["c"],signed[i]["m"])
    print(f"    {i+1}: sig={r['sig']} wm={r['wm']} ai={r['ai']} -> {r['status']} ({r['conf']})")

print("\n  Light edit:")
for i in range(3):
    ed=signed[i]["c"].replace("is","represents",1)
    r=p.full_verify(ed,signed[i]["m"])
    print(f"    {i+1}: sig={r['sig']} wm={r['wm']} ai={r['ai']} -> {r['status']} ({r['conf']})")

print("\n  Heavy tamper:")
for i,t in enumerate(["I love pizza on sunny days.","Stock market crashed yesterday.","My cat knocked over coffee."]):
    r=p.full_verify(t,signed[i]["m"])
    print(f"    {i+1}: sig={r['sig']} wm={r['wm']} ai={r['ai']} -> {r['status']} ({r['conf']})")


</details>
